In [1]:
# Global imports and configuration
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import glob
import json

from grid_processing import build_new_grid_xesmf, sort_ds, create_regridder

# Environment setup
print(f"Working directory: {os.getcwd()}")
print(f"Python executable: {sys.executable}")
os.environ['HOME'] = '/mnt/disk1/aiotlab/trieutq'

# Global grid parameters
LAT_START, LON_START = 20.027350, 104.790862
STEP_NEW = 0.02780
ROWS, COLS = 79, 127

# Global paths
BASE_DIR = os.path.expanduser('~/inest/eccad/data/qfed')
OUTPUT_DIR = os.path.join(BASE_DIR, 'json')
NC_DIR = os.path.join(BASE_DIR, 'nc')
DATA_FILE_PATT = "*.nc4"

print(f"Grid parameters: {ROWS}x{COLS}, step={STEP_NEW}")
print(f"Base directory: {BASE_DIR}")


Working directory: /mnt/disk1/aiotlab/trieutq/inest/eccad/handle
Python executable: /mnt/disk1/aiotlab/trieutq/.venv/bin/python
Grid parameters: 79x127, step=0.0278
Base directory: /mnt/disk1/aiotlab/trieutq/inest/eccad/data/qfed


In [ ]:
xr.open_dataset("~/inest/eccad/data/qfed/nc/qfed2.emis_acet.061.20230101.nc4")
print(xr)

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/disk1/aiotlab/trieutq/inest/eccad/handle/inest/eccad/data/qfed/nc/qfed2.emis_acet.061.20230101.nc4'

In [ ]:
# Batch process NC files to JSON
os.makedirs(OUTPUT_DIR, exist_ok=True)
nc_files = glob.glob(os.path.join(NC_DIR, DATA_FILE_PATT))

def ds_to_json(ds_new: xr.Dataset, max_rows: int = None) -> list[dict]:
    """Convert dataset to JSON format with row/col indices"""
    df = ds_new.to_dataframe().reset_index()

    # Create row/col indices
    lat_vals = df["lat"].unique()
    lon_vals = df["lon"].unique()
    lat_to_row = {v: i for i, v in enumerate(lat_vals)}
    lon_to_col = {v: i for i, v in enumerate(lon_vals)}

    df["row"] = df["lat"].map(lat_to_row)
    df["col"] = df["lon"].map(lon_to_col)

    if "sum" in df.columns:
        df["value"] = df["sum"]
    elif "emiss_bb" in df.columns:
        df["value"] = df["emiss_bb"]
    elif "emiss_bio" in df.columns:
        df["value"] = df["emiss_bio"]
    elif "all_sources" in df.columns:
        df["value"] = df["all_sources"]
    elif "biomass" in df.columns:
        df["value"] = df["biomass"]
    else:
        print(df)
        raise ValueError("Dataset does not contain required variables.")

    # Reorder columns
    df["time"] = df["time"].dt.strftime("%Y-%m-%d")

    df = df[["time", "row", "col", "lat", "lon", "value"]]

    if max_rows:
        df = df.iloc[:max_rows]

    return df.to_dict(orient="records")

regidder = None
for nc_file in nc_files:
    print(f"Processing {os.path.basename(nc_file)}")
    # Load and process dataset
    ds = sort_ds(xr.open_dataset(nc_file))
    if regidder is None:
        print("Creating regridder...")
        regidder = create_regridder(ds, lat_start=LAT_START, lon_start=LON_START, rows=ROWS, cols=COLS, step=STEP_NEW, method="conservative")
        print("Regridder created.")
    ds_new = regidder(ds)
    json_data = ds_to_json(ds_new)

    # Save JSON
    json_filename = os.path.splitext(os.path.basename(nc_file))[0] + '.json'
    json_path = os.path.join(OUTPUT_DIR, json_filename)

    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=2)

    print(f"Saved {json_path}")

Processing qfed2.emis_co2.061.20230827.nc4
Creating regridder...


In [ ]:
# Visualization comparison
sample_file = "~/inest/eccad/data/gfed4/nc/GFED4_Glb_0.25x0.25_bb_BC__daily_2022.nc"
sample_file = "~/inest/eccad/data/ant2/nc/CAMS-GLOB-ANT_Glb_0.1x0.1_anthro_bc_v5.3_monthly_2023.nc"
sample_file = "~/inest/eccad/data/bio2/nc/CAMS-GLOB-BIO_Glb_0.25x0.25_bio_acetaldehyde_v3.1_monthly_2023.nc"
sample_file = "~/inest/eccad/handle/resources/qfed2.emis_acet.061.20230101.nc4"
ds = sort_ds(xr.open_dataset(sample_file))
print("Original dataset:")
print(ds)


In [ ]:
ds_new = build_new_grid_xesmf(ds, lat_start=LAT_START, lon_start=LON_START, rows=ROWS, cols=COLS, step=STEP_NEW, method="patch")
print("\nNew grid dataset:")
print(ds_new)

In [ ]:
ds_new2 = build_new_grid_xesmf(ds, lat_start=LAT_START, lon_start=LON_START, rows=ROWS, cols=COLS, step=STEP_NEW, method="conservative")
print("\nNew grid dataset (conservative):")
print(ds_new2)

In [ ]:
# Visualization
LAT_MIN, LAT_MAX = 21, 22
LON_MIN, LON_MAX = 107, 108
var = "biomass"

# Plot comparison
fig, axes = plt.subplots(3, 2, figsize=(16, 12), constrained_layout=True)
axes = axes.flatten()

print(ds.lat.values[:3], ds.lat.values[-3:])  # xem chiều lat
print("Requested box:",
      "lat", (LAT_MIN, LAT_MAX), "lon", (LON_MIN, LON_MAX))
print("Dataset ranges:",
      "lat", (float(ds.lat.max()), float(ds.lat.min())),
      "lon", (float(ds.lon.min()), float(ds.lon.max())))

# Cut to region of interest
ds_cut = ds.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
ds_cut[var].isel(time=0).plot.imshow(
    ax=axes[0], add_colorbar=True, cmap="viridis"
)

ds_cut["biomass_tf"].isel(time=0).plot.imshow(
    ax=axes[2], add_colorbar=True, cmap="viridis"
)

ds_cut["biomass_xf"].isel(time=0).plot.imshow(
    ax=axes[3], add_colorbar=True, cmap="viridis"
)

ds_cut["biomass_sv"].isel(time=0).plot.imshow(
    ax=axes[4], add_colorbar=True, cmap="viridis"
)

ds_cut["biomass_gl"].isel(time=0).plot.imshow(
    ax=axes[5], add_colorbar=True, cmap="viridis"
)


# # New grid
# ds_new_cut = ds_new.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
# ds_new_cut[var].isel(time=0).plot.imshow(
#     ax=axes[2], add_colorbar=True, cmap="viridis"
# )

# ds_new2_cut = ds_new2.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX))
# ds_new2_cut[var].isel(time=0).plot.imshow(
#     ax=axes[3], add_colorbar=True, cmap="viridis"
# )


plt.suptitle("Grid Comparison", fontsize=14)
plt.show()

In [ ]:
# Hiển thị dữ liệu lưới trên bản đồ OpenStreetMap với folium
import folium
from folium.plugins import HeatMap

# Chuyển dữ liệu thành DataFrame
df = ds_new_cut['sum'].isel(time=0).to_dataframe().reset_index()

# Tạo bản đồ trung tâm vùng quan tâm
center_lat = (LAT_MIN + LAT_MAX) / 2
center_lon = (LON_MIN + LON_MAX) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

# Tạo danh sách điểm heatmap: [lat, lon, value]
heat_data = [[row['lat'], row['lon'], row['sum']] for _, row in df.iterrows() if not np.isnan(row['sum'])]

HeatMap(heat_data, radius=8, blur=15, max_zoom=1).add_to(m)

# Hiển thị bản đồ
m